> **Niveau 🟢 facile — le code est écrit, complétez les `___`**

# Notebook 1 — De l'ADN du patient à la protéine

On dispose de la séquence codante de **quatre gènes exprimés dans le globule rouge**, chez un
sujet de référence et chez le patient :

| Gène | Protéine |
|---|---|
| `HBA1` | alpha-globine |
| `HBB` | bêta-globine |
| `HBD` | delta-globine |
| `HBG1` | gamma-globine (hémoglobine fœtale) |

Objectif : trouver **lequel de ces gènes est muté chez le patient**, puis suivre l'effet de
cette mutation à travers le dogme central **ADN → ARN → protéine**, jusqu'à l'acide aminé
modifié.

**Mode d'emploi**
- `Maj + Entrée` exécute une cellule et passe à la suivante.
- Exécutez les cellules **dans l'ordre**, de haut en bas.
- Les cellules **✔️ Vérification** ne se modifient pas : elles affichent ✅ quand votre code est juste.

## 0. Charger les séquences

Exécutez la cellule ci-dessous : elle crée deux fichiers FASTA — la référence et le patient —
et range leur contenu dans deux dictionnaires, `REFERENCE` et `PATIENT`.

In [ ]:
#@title ▶️ Exécutez cette cellule pour charger les séquences (ne pas modifier)
# Séquences codantes de référence (RefSeq) de quatre gènes exprimés dans le globule rouge,
# et les mêmes gènes séquencés chez le patient.

FASTA_REFERENCE = """>HBA1 alpha-globine — NM_000558.5
ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAGGTCGGCGCG
CACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCCTGTCCTTCCCCACCACC
AAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCACGGC
AAGAAGGTGGCCGACGCGCTGACCAACGCCGTGGCGCACGTGGACGACATGCCCAACGCG
CTGTCCGCCCTGAGCGACCTGCACGCGCACAAGCTTCGGGTGGACCCGGTCAACTTCAAG
CTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCT
GCGGTGCACGCCTCCCTGGACAAGTTCCTGGCTTCTGTGAGCACCGTGCTGACCTCCAAA
TACCGTTAA
>HBB bêta-globine — NM_000518.5
ATGGTGCATCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAAC
GTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGC
AAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCCCACAAGTATCACTAA
>HBD delta-globine — NM_000519.4
ATGGTGCATCTGACTCCTGAGGAGAAGACTGCTGTCAATGCCCTGTGGGGCAAAGTGAAC
GTGGATGCAGTTGGTGGTGAGGCCCTGGGCAGATTACTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCTCTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAGGTGCTAGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACTTTTTCTCAGCTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCTTGGGCAATGTGCTGGTGTGTGTGCTGGCCCGCAACTTTGGC
AAGGAATTCACCCCACAAATGCAGGCTGCCTATCAGAAGGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCTCACAAGTACCATTGA
>HBG1 gamma-globine (hémoglobine fœtale) — NM_000559.3
ATGGGTCATTTCACAGAGGAGGACAAGGCTACTATCACAAGCCTGTGGGGCAAGGTGAAT
GTGGAAGATGCTGGAGGAGAAACCCTGGGAAGGCTCCTGGTTGTCTACCCATGGACCCAG
AGGTTCTTTGACAGCTTTGGCAACCTGTCCTCTGCCTCTGCCATCATGGGCAACCCCAAA
GTCAAGGCACATGGCAAGAAGGTGCTGACTTCCTTGGGAGATGCCACAAAGCACCTGGAT
GATCTCAAGGGCACCTTTGCCCAGCTGAGTGAACTGCACTGTGACAAGCTGCATGTGGAT
CCTGAGAACTTCAAGCTCCTGGGAAATGTGCTGGTGACCGTTTTGGCAATCCATTTCGGC
AAAGAATTCACCCCTGAGGTGCAGGCTTCCTGGCAGAAGATGGTGACTGCAGTGGCCAGT
GCCCTGTCCTCCAGATACCACTGA
"""

FASTA_PATIENT = """>HBA1 alpha-globine — NM_000558.5
ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAGGTCGGCGCG
CACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCCTGTCCTTCCCCACCACC
AAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCACGGC
AAGAAGGTGGCCGACGCGCTGACCAACGCCGTGGCGCACGTGGACGACATGCCCAACGCG
CTGTCCGCCCTGAGCGACCTGCACGCGCACAAGCTTCGGGTGGACCCGGTCAACTTCAAG
CTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCT
GCGGTGCACGCCTCCCTGGACAAGTTCCTGGCTTCTGTGAGCACCGTGCTGACCTCCAAA
TACCGTTAA
>HBB bêta-globine — NM_000518.5
ATGGTGCATCTGACTCCTGTGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAAC
GTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGC
AAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCCCACAAGTATCACTAA
>HBD delta-globine — NM_000519.4
ATGGTGCATCTGACTCCTGAGGAGAAGACTGCTGTCAATGCCCTGTGGGGCAAAGTGAAC
GTGGATGCAGTTGGTGGTGAGGCCCTGGGCAGATTACTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCTCTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAGGTGCTAGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACTTTTTCTCAGCTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCTTGGGCAATGTGCTGGTGTGTGTGCTGGCCCGCAACTTTGGC
AAGGAATTCACCCCACAAATGCAGGCTGCCTATCAGAAGGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCTCACAAGTACCATTGA
>HBG1 gamma-globine (hémoglobine fœtale) — NM_000559.3
ATGGGTCATTTCACAGAGGAGGACAAGGCTACTATCACAAGCCTGTGGGGCAAGGTGAAT
GTGGAAGATGCTGGAGGAGAAACCCTGGGAAGGCTCCTGGTTGTCTACCCATGGACCCAG
AGGTTCTTTGACAGCTTTGGCAACCTGTCCTCTGCCTCTGCCATCATGGGCAACCCCAAA
GTCAAGGCACATGGCAAGAAGGTGCTGACTTCCTTGGGAGATGCCACAAAGCACCTGGAT
GATCTCAAGGGCACCTTTGCCCAGCTGAGTGAACTGCACTGTGACAAGCTGCATGTGGAT
CCTGAGAACTTCAAGCTCCTGGGAAATGTGCTGGTGACCGTTTTGGCAATCCATTTCGGC
AAAGAATTCACCCCTGAGGTGCAGGCTTCCTGGCAGAAGATGGTGACTGCAGTGGCCAGT
GCCCTGTCCTCCAGATACCACTGA
"""

with open("reference.fasta", "w") as f:
    f.write(FASTA_REFERENCE)
with open("patient.fasta", "w") as f:
    f.write(FASTA_PATIENT)


def lire_multifasta(nom_fichier):
    """Lit un fichier FASTA contenant plusieurs séquences.
    Renvoie un dictionnaire {nom du gène: séquence}."""
    sequences = {}
    nom = None
    with open(nom_fichier) as f:
        for ligne in f:
            ligne = ligne.strip()
            if ligne.startswith(">"):
                nom = ligne[1:].split()[0]
                sequences[nom] = ""
            elif ligne:
                sequences[nom] = sequences[nom] + ligne
    return sequences


REFERENCE = lire_multifasta("reference.fasta")
PATIENT = lire_multifasta("patient.fasta")
print(len(REFERENCE), "gènes chargés ✅")

Le format **FASTA** : une ligne d'en-tête qui commence par `>`, puis la séquence, coupée en
lignes de 60 lettres. Un même fichier peut contenir plusieurs séquences à la suite — ici, les
quatre gènes. Voici le fichier de référence tel qu'il est écrit sur le disque :

In [ ]:
print(FASTA_REFERENCE)

## 1. Les quatre gènes

`REFERENCE` et `PATIENT` sont des **dictionnaires** : à chaque nom de gène correspond une
séquence. Affichez le nom de chaque gène et la longueur de sa séquence.

In [ ]:
# REFERENCE est un dictionnaire : .items() donne les paires (nom du gène, séquence)
for nom, sequence in REFERENCE.items():
    # afficher le nom du gène, puis la longueur de sa séquence avec len()
    print(nom, len(___))

# len() sur un dictionnaire donne son nombre de clés
print("nombre de gènes :", len(___))

## 2. Quel gène est muté chez le patient ?

Les quatre gènes du patient ont été séquencés. Trois sont identiques à la référence, un seul
diffère — c'est celui-là qu'il faut trouver.

Deux chaînes de caractères se comparent directement : `"ACGT" == "ACGT"` vaut `True`.

In [ ]:
gene_mute = None

for nom in REFERENCE:
    sequence_ref = REFERENCE[nom]
    # la séquence du même gène chez le patient
    sequence_patient = ___[nom]
    if sequence_ref == sequence_patient:
        print(nom, ": identique")
    else:
        print(nom, ": DIFFÉRENT")
        # retenir le nom de ce gène
        gene_mute = ___

print()
print("gène muté :", gene_mute)

# tout le reste du notebook portera sur ce gène
adn_wt = REFERENCE[gene_mute]
adn_patient = PATIENT[gene_mute]

In [ ]:
# ✔️ Vérification — exécutez sans modifier
assert gene_mute == "HBB", "ce n'est pas le bon gène : recomparez chaque paire de séquences"
assert adn_wt == REFERENCE["HBB"] and adn_patient == PATIENT["HBB"], \
    "adn_wt et adn_patient doivent être les séquences du gène muté"
print("✅ le gène muté est HBB — la bêta-globine, une des deux chaînes de l'hémoglobine")

## 3. Comparer les deux séquences, base par base

À l'œil, 444 lettres, c'est trop. On écrit une fonction `trouver_mutations(ref, patient)` qui
parcourt les deux séquences position par position et renvoie la **liste des différences**,
chacune sous la forme `(index, base_ref, base_patient)`.

Exemple : `trouver_mutations("ACGT", "ACCT")` → `[(2, "G", "C")]`

⚠️ Python numérote à partir de **0** ; les biologistes numérotent les nucléotides à partir de **1**.
Le nucléotide n°1 est à l'index 0.

In [ ]:
def trouver_mutations(ref, patient):
    # une liste vide, qu'on remplira avec les différences trouvées
    mutations = []
    # parcourir tous les index i, de 0 à len(ref) exclu
    for i in range(___):
        # condition : la base de ref à l'index i est différente (!=) de celle du patient
        if ___:
            # ajouter le triplet (index, base de ref, base du patient)
            mutations.append((i, ref[i], ___))
    return mutations


mutations = trouver_mutations(adn_wt, adn_patient)
for index, base_ref, base_patient in mutations:
    # numéro du nucléotide pour un biologiste = index Python + 1
    numero = ___
    print(f"index Python {index} = nucléotide n°{numero} : {base_ref} → {base_patient}")

In [ ]:
# ✔️ Vérification — exécutez sans modifier
assert trouver_mutations("ACGT", "ACCT") == [(2, "G", "C")], "trouver_mutations('ACGT', 'ACCT') doit donner [(2, 'G', 'C')]"
assert trouver_mutations("AAAA", "AAAA") == [], "deux séquences identiques : la liste doit être vide"
assert len(mutations) == 1, "le patient doit avoir exactement 1 différence avec la référence"
print("✅ Mutation trouvée :", mutations)

## 4. Transcrire l'ADN en ARN

La séquence fournie est celle du **brin codant**. L'ARN messager a la même séquence que le
brin codant, à une différence près : la thymine **T** est remplacée par l'uracile **U**.

Écrivez `transcrire(adn)` qui renvoie l'ARN correspondant.

Exemple : `transcrire("ATGGAG")` → `"AUGGAG"`

In [ ]:
def transcrire(adn):
    # la méthode .replace(ancien, nouveau) remplace un caractère par un autre dans une chaîne
    # ici : remplacer "T" par "U"
    return adn.replace(___, ___)


arn_wt = transcrire(adn_wt)
# même chose pour l'ADN du patient
arn_patient = transcrire(___)
print(arn_wt)

In [ ]:
# ✔️ Vérification — exécutez sans modifier
assert transcrire("ATGGAG") == "AUGGAG", "transcrire('ATGGAG') doit donner 'AUGGAG'"
assert "T" not in arn_wt and "T" not in arn_patient, "il reste des T dans l'ARN"
assert arn_wt.startswith("AUG"), "l'ARNm doit commencer par le codon start AUG"
print("✅ Transcription correcte. L'ARNm commence par", arn_wt[:3])

## 5. Découper l'ARN en codons

Le ribosome lit l'ARN **trois bases à la fois**, à partir du codon start : c'est le cadre de
lecture. Écrivez `codons(arn)` qui renvoie la liste des triplets.

Exemple : `codons("AUGGAGUAA")` → `["AUG", "GAG", "UAA"]`

Rappel : `arn[0:3]` donne les caractères d'index 0, 1 et 2.

In [ ]:
def codons(arn):
    liste = []
    # range(debut, fin, pas) : les index 0, 3, 6, 9... → le pas est la taille d'un codon
    for i in range(0, len(arn), ___):
        # le codon qui commence à l'index i : de i (inclus) à i + 3 (exclu)
        liste.append(arn[i:___])
    return liste


codons_wt = codons(arn_wt)
codons_patient = codons(arn_patient)
print("nombre de codons :", len(codons_wt))

# dans quel codon tombe la mutation ?
# index du nucléotide muté (trouvé à l'exercice 2)
index_mutation = mutations[0][0]
# chaque codon contient 3 nucléotides : on divise l'index par 3 (division entière //)
index_codon = index_mutation // ___
print(f"la mutation est dans le codon n°{index_codon + 1} (index Python {index_codon})")
print("codon WT      :", codons_wt[index_codon])
# le même codon chez le patient
print("codon patient :", ___)

In [ ]:
# ✔️ Vérification — exécutez sans modifier
assert codons("AUGGAGUAA") == ["AUG", "GAG", "UAA"], "codons('AUGGAGUAA') doit donner ['AUG', 'GAG', 'UAA']"
assert len(codons_wt) == 148, "l'ARN de HBB contient 148 codons"
assert index_codon == 6, "la mutation est dans le 7e codon, donc à l'index 6"
assert codons_wt[index_codon] == "GAG" and codons_patient[index_codon] == "GUG"
print("✅ Codon muté :", codons_wt[index_codon], "→", codons_patient[index_codon])

## 6. Traduire l'ARN en protéine

Voici le code génétique sous forme de dictionnaire : à chaque codon, il associe un acide aminé
(code à une lettre). Les trois codons stop sont notés `"*"`. Exécutez la cellule.

In [ ]:
CODE_GENETIQUE = {
    "UUU": "F", "UUC": "F", "UUA": "L", "UUG": "L",
    "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S",
    "UAU": "Y", "UAC": "Y", "UAA": "*", "UAG": "*",
    "UGU": "C", "UGC": "C", "UGA": "*", "UGG": "W",
    "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
    "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "CAU": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AUU": "I", "AUC": "I", "AUA": "I", "AUG": "M",
    "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "AAU": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "AGU": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V",
    "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "GAU": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}

print(CODE_GENETIQUE["AUG"], CODE_GENETIQUE["GAG"], CODE_GENETIQUE["UAA"])

Écrivez `traduire(arn)` : pour chaque codon, on ajoute l'acide aminé correspondant à la
protéine ; **au premier codon stop, on s'arrête** (le stop n'est pas ajouté).

Exemple : `traduire("AUGGAGUAAGGG")` → `"ME"`

Utilisez votre fonction `codons()` de l'exercice 4.

In [ ]:
def traduire(arn):
    # la protéine commence vide
    proteine = ""
    # parcourir la liste des codons de l'ARN (fonction de l'exercice 4)
    for codon in ___:
        # chercher l'acide aminé du codon dans le dictionnaire CODE_GENETIQUE
        acide_amine = CODE_GENETIQUE[___]
        # codon stop : on sort de la boucle
        if acide_amine == "*":
            break
        # sinon : ajouter l'acide aminé au bout de la protéine
        proteine = proteine + ___
    return proteine


proteine_wt = traduire(arn_wt)
# traduire l'ARN du patient
proteine_patient = traduire(___)
print(proteine_wt)
print("longueur :", len(proteine_wt), "acides aminés")

In [ ]:
# ✔️ Vérification — exécutez sans modifier
assert traduire("AUGGAGUAAGGG") == "ME", "traduire('AUGGAGUAAGGG') doit donner 'ME' (arrêt au stop)"
assert proteine_wt.startswith("MVHLTPEEK"), "la bêta-globine commence par MVHLTPEEK"
assert len(proteine_wt) == 147, "la bêta-globine compte 147 acides aminés (méthionine initiale comprise)"
assert len(proteine_patient) == len(proteine_wt), "les deux protéines ont la même longueur"
print("✅ Traduction correcte :", len(proteine_wt), "acides aminés")

## 7. Quel acide aminé a changé ?

Une protéine, comme l'ADN, est une chaîne de caractères : la fonction `trouver_mutations()`
de l'exercice 2 marche donc aussi sur les protéines.

Le dictionnaire `NOMS` donne, pour chaque lettre, l'abréviation à trois lettres et le nom.

In [ ]:
NOMS = {
    "A": ("Ala", "alanine"),
    "R": ("Arg", "arginine"),
    "N": ("Asn", "asparagine"),
    "D": ("Asp", "aspartate"),
    "C": ("Cys", "cystéine"),
    "Q": ("Gln", "glutamine"),
    "E": ("Glu", "glutamate"),
    "G": ("Gly", "glycine"),
    "H": ("His", "histidine"),
    "I": ("Ile", "isoleucine"),
    "L": ("Leu", "leucine"),
    "K": ("Lys", "lysine"),
    "M": ("Met", "méthionine"),
    "F": ("Phe", "phénylalanine"),
    "P": ("Pro", "proline"),
    "S": ("Ser", "sérine"),
    "T": ("Thr", "thréonine"),
    "W": ("Trp", "tryptophane"),
    "Y": ("Tyr", "tyrosine"),
    "V": ("Val", "valine"),
}

print(NOMS["M"])

In [ ]:
# réutiliser trouver_mutations sur les deux protéines
differences = trouver_mutations(___, ___)

for index, aa_wt, aa_patient in differences:
    # numéro de l'acide aminé = index Python + 1
    numero = ___
    print(f"position {numero} : {aa_wt} → {aa_patient}")
    # NOMS[lettre] donne (abréviation, nom) : [1] pour le nom
    print(f"  {NOMS[aa_wt][1]} → {NOMS[___][1]}")

In [ ]:
# ✔️ Vérification — exécutez sans modifier
assert differences == [(6, "E", "V")], "une seule différence attendue, à l'index 6 : E → V"
print("✅ Diagnostic :", NOMS["E"][0], "→", NOMS["V"][0], "en position 7 de la chaîne traduite")

## Bilan

| Niveau | Référence (WT) | Patient |
|---|---|---|
| Gène muté parmi les quatre | — | `HBB`, la bêta-globine |
| ADN, nucléotide n°20 | `A` | `T` |
| ARN, codon n°7 | `GAG` | `GUG` |
| Protéine, acide aminé n°7 | E — glutamate | V — valine |

Une seule base modifiée sur 444 remplace un acide aminé chargé (glutamate) par un acide
aminé hydrophobe (valine).

**Une simplification à connaître.** Ici, les trois autres gènes du patient sont strictement
identiques à la référence. Chez un individu réel, chacun porterait des dizaines de variants
sans conséquence : la difficulté n'est pas de trouver *une* différence, c'est de repérer
celle qui change la protéine.

**Pourquoi la littérature médicale dit « Glu6Val » et pas « Glu7Val » ?** Dans la protéine
mature, la méthionine initiale (codée par le codon start `AUG`) est retirée. La numérotation
historique de la bêta-globine commence donc à l'acide aminé suivant : le glutamate n°7 de
notre chaîne traduite y porte le n°6.

Trois numérotations pour le même acide aminé :
- index Python : **6** ;
- position dans la chaîne traduite (méthionine comprise) : **7** ;
- position dans la protéine mature (numérotation clinique) : **6**.